1) Imports and configuration

In [1]:
import requests
import pandas as pd
import time

from bs4 import BeautifulSoup
from urllib.parse import urljoin
from tqdm import tqdm

What each library does:
- requests: sends HTTP requests to the website.
- BeautifulSoup: reads and extracts information from HTML.
- pandas: stores the extracted products in a DataFrame.
- time: adds pauses between requests.
- urljoin: converts relative URLs into complete URLs.
- tqdm: displays a progress bar.

2) Website configuration

In [2]:
BASE_URL = "https://www.pharmaly-dz.com"

HEADERS = {
    "User-Agent": "Academic research scraper (authorized collection)"
}

session = requests.Session()
session.headers.update(HEADERS)


3) Category configuration

In [3]:
CATEGORIES = {
    "Sante": f"{BASE_URL}/categorie/sante/",
    "Visage": f"{BASE_URL}/categorie/visage/",
    "Corps": f"{BASE_URL}/categorie/corps/",
    "Cheveux": f"{BASE_URL}/categorie/cheveux/",
    "K-Beauty": urljoin(BASE_URL, "/categorie/k-beauty/"),
    "Solaire": f"{BASE_URL}/categorie/solaire/",
    "Hygiene": f"{BASE_URL}/categorie/hygiene/",
    "Maquillage": f"{BASE_URL}/categorie/maquillage-et-parfums/"
}

print("Number of categories:", len(CATEGORIES))

Number of categories: 8


In [4]:
def get_category_product_urls(category_url):

    try:
        response = session.get(category_url, timeout=(10, 60))
        response.raise_for_status()

        soup = BeautifulSoup(response.text, "lxml")

        links = soup.select("a.jet-listing-dynamic-image__link")

        product_urls = []

        for link in links:
            url = link.get("href")

            if url and "/produit/" in url:
                product_urls.append(urljoin(BASE_URL, url.strip()))

        # Remove duplicates while preserving order
        product_urls = list(dict.fromkeys(product_urls))

        return product_urls

    except requests.RequestException as error:
        print(f"Error accessing {category_url}: {error}")
        return []

In [5]:
def scrape_product_page(url, categories):

    try:
        response = session.get(url, timeout=(10, 60))
        response.raise_for_status()

        soup = BeautifulSoup(response.text, "lxml")

        # Product name
        title = soup.select_one("h1")

        # Brand
        brand = soup.select_one(
            ".jet-listing-dynamic-terms__link"
        )

        # Price
        price = soup.select_one(
            ".elementor-widget-woocommerce-product-price p.price"
        )

        original_price = None
        current_price = None

        if price:

            old_price = price.select_one("del")
            new_price = price.select_one("ins")

            if old_price and new_price:
                original_price = old_price.get_text(" ", strip=True)
                current_price = new_price.get_text(" ", strip=True)

            else:
                current_price = price.get_text(" ", strip=True)

        # Short description
        short_description = soup.select(
            ".elementor-widget-text-editor p"
        )

        short_description = " ".join(
            p.get_text(" ", strip=True)
            for p in short_description
        )

        # Full description
        full_description = soup.select_one(
            ".elementor-tab-content[data-tab='1']"
        )

        # Main product image
        image = soup.select_one(
            ".jet-woo-product-gallery__image img"
        )

        image_url = None

        if image:
            image_source = (
                image.get("src")
                or image.get("data-src")
                or image.get("data-lazy-src")
            )

            if image_source:
                image_url = urljoin(BASE_URL, image_source)

        return {
            "product_name": title.get_text(" ", strip=True) if title else None,
            "brand": brand.get_text(" ", strip=True) if brand else None,
            "categories": ", ".join(categories),
            "original_price": original_price,
            "current_price": current_price,
            "short_description": short_description or None,
            "full_description": (
                full_description.get_text(" ", strip=True)
                if full_description else None
            ),
            "image_url": image_url,
            "product_url": url
        }

    except requests.RequestException as error:
        print(f"Failed: {url} | {error}")
        return None

In [7]:

all_product_urls = {}

for category, category_url in CATEGORIES.items():

    print(f"\nScraping category: {category}")

    urls = get_category_product_urls(category_url)

    for url in urls:
        if url not in all_product_urls:
            all_product_urls[url] = []

        if category not in all_product_urls[url]:
            all_product_urls[url].append(category)

    print("Unique products:", len(urls))

    time.sleep(1)

print("\nTotal unique product URLs:", len(all_product_urls))


Scraping category: Sante
Unique products: 12

Scraping category: Visage
Unique products: 19

Scraping category: Corps
Unique products: 13

Scraping category: Cheveux
Unique products: 20

Scraping category: K-Beauty
Unique products: 20

Scraping category: Solaire
Unique products: 20

Scraping category: Hygiene
Unique products: 16

Scraping category: Maquillage
Unique products: 14

Total unique product URLs: 118


In [10]:
url = CATEGORIES["Visage"]

response = session.get(url)
soup = BeautifulSoup(response.text, "lxml")

cards = soup.select("div.jet-listing-grid__item")
links = soup.select("a.jet-listing-dynamic-image__link")

print("Product cards in initial HTML:", len(cards))
print("Product links in initial HTML:", len(links))
print("Unique product URLs:", len(get_category_product_urls(url)))

Product cards in initial HTML: 27
Product links in initial HTML: 20
Unique product URLs: 19


In [11]:
for category, url in CATEGORIES.items():

    response = session.get(url, timeout=60)
    soup = BeautifulSoup(response.text, "lxml")

    grids = soup.select("div.jet-listing-grid")

    print(f"\n{'='*40}")
    print("CATEGORY:", category)
    print("Product grids:", len(grids))

    for i, grid in enumerate(grids, 1):

        cards = grid.select("div.jet-listing-grid__item")
        links = grid.select("a.jet-listing-dynamic-image__link")

        print(f"Grid {i}:")
        print("  Cards:", len(cards))
        print("  Links:", len(links))

    print("Initial unique URLs:",
          len(get_category_product_urls(url)))


CATEGORY: Sante
Product grids: 3
Grid 1:
  Cards: 2
  Links: 0
Grid 2:
  Cards: 2
  Links: 2
Grid 3:
  Cards: 10
  Links: 10
Initial unique URLs: 12

CATEGORY: Visage
Product grids: 3
Grid 1:
  Cards: 7
  Links: 0
Grid 2:
  Cards: 10
  Links: 10
Grid 3:
  Cards: 10
  Links: 10
Initial unique URLs: 19

CATEGORY: Corps
Product grids: 3
Grid 1:
  Cards: 6
  Links: 0
Grid 2:
  Cards: 3
  Links: 3
Grid 3:
  Cards: 10
  Links: 10
Initial unique URLs: 13

CATEGORY: Cheveux
Product grids: 3
Grid 1:
  Cards: 8
  Links: 0
Grid 2:
  Cards: 10
  Links: 10
Grid 3:
  Cards: 10
  Links: 10
Initial unique URLs: 20

CATEGORY: K-Beauty
Product grids: 3
Grid 1:
  Cards: 1
  Links: 0
Grid 2:
  Cards: 10
  Links: 10
Grid 3:
  Cards: 10
  Links: 10
Initial unique URLs: 20

CATEGORY: Solaire
Product grids: 3
Grid 1:
  Cards: 3
  Links: 0
Grid 2:
  Cards: 10
  Links: 10
Grid 3:
  Cards: 10
  Links: 10
Initial unique URLs: 20

CATEGORY: Hygiene
Product grids: 3
Grid 1:
  Cards: 5
  Links: 0
Grid 2:
  Cards: 6

In [13]:
import time
import requests

time.sleep(10)

try:
    response = session.get(
        CATEGORIES["Visage"],
        timeout=(10, 60)
    )
    response.raise_for_status()

    print("Connection successful:", response.status_code)

except requests.exceptions.RequestException as e:
    print("Connection failed:", e)

Connection successful: 200


In [14]:
category_results = {}

for category, url in CATEGORIES.items():

    print(f"\nChecking {category}...")

    try:
        response = session.get(url, timeout=(10, 60))
        response.raise_for_status()

        soup = BeautifulSoup(response.text, "lxml")
        grids = soup.select("div.jet-listing-grid")

        category_results[category] = []

        for grid_number, grid in enumerate(grids, 1):

            cards = grid.select("div.jet-listing-grid__item")

            for card_number, card in enumerate(cards, 1):

                print(
                    f"Grid {grid_number}, Card {card_number}:",
                    card.get_text(" ", strip=True)[:150]
                )

        print("Page inspected successfully.")

    except requests.exceptions.RequestException as e:
        print(f"Could not inspect {category}: {e}")

    time.sleep(5)


Checking Sante...
Grid 1, Card 1: Compléments Alimentaires
Grid 1, Card 2: Sexualité
Grid 2, Card 1: Laboratoires NHCO Nutrition Effluvium Croissance Vitalité Du Cheveu 168 Gel Promo -1600 DA DA 11.500,00 Le prix initial était : DA 11.500,00. DA 9.900
Grid 2, Card 2: Forcapil Forcapil Anti Chute 90 Comprimés Coming soon Promo -800 DA DA 9.000,00 Le prix initial était : DA 9.000,00. DA 8.200,00 Le prix actuel est : 
Grid 3, Card 1: Biogaia Biogaia Gouttesprotectis   10ml Nouveau DA 5.750,00 J'ACHÈTE
Grid 3, Card 2: Fortimel Fortimel Saveur Peche Mangue 2.4 Kcal   200ml Nouveau DA 740,00 J'ACHÈTE
Grid 3, Card 3: Laboratoires ACM Acm Novophane Complément Alimentaire Ongles/cheveux B/180   108g DA 6.500,00 J'ACHÈTE
Grid 3, Card 4: Cheks Cheks Postbiotic Balance  Bt/45 DA 2.700,00 J'ACHÈTE
Grid 3, Card 5: Granions Granions Biotine Vitamine B8 46g DA 4.800,00 J'ACHÈTE
Grid 3, Card 6: Byocore Byocore Marin Collagène 60caps   27g Coming soon DA 3.500,00 Précommander
Grid 3, Card 7: Nature's B

In [8]:
test_urls = list(all_product_urls.keys())[:3]

test_results = []

for url in test_urls:

    categories = all_product_urls[url]

    product = scrape_product_page(url, categories)

    if product:
        test_results.append(product)

    time.sleep(1)

test_df = pd.DataFrame(test_results)

display(test_df[[
    "product_name",
    "brand",
    "current_price",
    "full_description"
]])

,product_name,brand,current_price,full_description
0,Effluvium Croissance Vitalité Du Cheveu 168 Gel,Laboratoires NHCO Nutrition,"DA 9.900,00",Le Effluvium Croissance Vitalité du Cheveu 168...
1,Forcapil Anti Chute 90 Comprimés,Forcapil,"DA 8.200,00",Forcapil Anti-Chute est un complément alimenta...
2,Biogaia Gouttesprotectis 10ml,Biogaia,"DA 5.750,00",BIOGAIA GOUTTESPROTECTIS 10ML


In [9]:
all_products = []

failed_urls = []

for url, categories in tqdm(
    all_product_urls.items(),
    desc="Scraping product pages"
):

    product = scrape_product_page(url, categories)

    if product is not None:
        all_products.append(product)
    else:
        failed_urls.append(url)

    time.sleep(1)

df = pd.DataFrame(all_products)

print("Successfully scraped:", len(df))
print("Failed requests:", len(failed_urls))

display(df.head())

Scraping product pages: 100%|██████████| 118/118 [26:17<00:00, 13.37s/it]

Successfully scraped: 118
Failed requests: 0


,product_name,brand,categories,original_price,current_price,short_description,full_description,image_url,product_url
0,Effluvium Croissance Vitalité Du Cheveu 168 Gel,Laboratoires NHCO Nutrition,"Sante, Cheveux","DA 11.500,00","DA 9.900,00",Complément capillaire qui stimule la croissanc...,Le Effluvium Croissance Vitalité du Cheveu 168...,https://www.pharmaly-dz.com/wp-content/uploads...,https://www.pharmaly-dz.com/produit/effluvium-...
1,Forcapil Anti Chute 90 Comprimés,Forcapil,Sante,"DA 9.000,00","DA 8.200,00",Complément anti-chute qui fortifie les cheveux...,Forcapil Anti-Chute est un complément alimenta...,https://www.pharmaly-dz.com/wp-content/uploads...,https://www.pharmaly-dz.com/produit/forcapil-a...
2,Biogaia Gouttesprotectis 10ml,Biogaia,Sante,NaN,"DA 5.750,00",NaN,BIOGAIA GOUTTESPROTECTIS 10ML,https://www.pharmaly-dz.com/wp-content/uploads...,https://www.pharmaly-dz.com/produit/biogaia-go...
3,Fortimel Saveur Peche Mangue 2.4 Kcal 200ml,Fortimel,Sante,NaN,"DA 740,00",Fortimel Protein 2.4 kcal Pêche Mangue est un ...,Le Fortimel Protein 2.4 kcal Saveur Pêche Mang...,https://www.pharmaly-dz.com/wp-content/uploads...,https://www.pharmaly-dz.com/produit/fortimel-s...
4,Acm Novophane Complément Alimentaire Ongles/ch...,Laboratoires ACM,Sante,NaN,"DA 6.500,00",Une crème corps apaisante et nourrissante qui ...,La crème apaisante ACM Sédacalm 120ml est spéc...,https://www.pharmaly-dz.com/wp-content/uploads...,https://www.pharmaly-dz.com/produit/acm-novoph...


In [15]:
import time
import requests
from bs4 import BeautifulSoup

category_results = {}

for category, url in CATEGORIES.items():

    print(f"\n{'='*40}")
    print(f"Checking: {category}")

    try:
        response = session.get(url, timeout=(10, 60))
        response.raise_for_status()

        soup = BeautifulSoup(response.text, "lxml")
        grids = soup.select("div.jet-listing-grid")

        category_results[category] = []

        for grid_number, grid in enumerate(grids, 1):

            cards = grid.select("div.jet-listing-grid__item")

            for card_number, card in enumerate(cards, 1):

                links = card.select("a[href]")

                for link in links:
                    href = link.get("href")

                    if href and "/produit/" in href:
                        category_results[category].append(
                            urljoin(BASE_URL, href)
                        )

        category_results[category] = list(
            dict.fromkeys(category_results[category])
        )

        print("Grids:", len(grids))
        print("Unique product URLs:", len(category_results[category]))

    except requests.exceptions.RequestException as e:
        print(f"Failed: {e}")

    time.sleep(5)


Checking: Sante
Grids: 3
Unique product URLs: 12

Checking: Visage
Grids: 3
Unique product URLs: 19

Checking: Corps
Grids: 3
Unique product URLs: 13

Checking: Cheveux
Grids: 3
Unique product URLs: 20

Checking: K-Beauty
Grids: 3
Unique product URLs: 20

Checking: Solaire
Grids: 3
Unique product URLs: 20

Checking: Hygiene
Grids: 3
Unique product URLs: 16

Checking: Maquillage
Grids: 3
Unique product URLs: 14


In [18]:
url = CATEGORIES["Sante"]

response = session.get(url, timeout=60)
soup = BeautifulSoup(response.text, "lxml")

for grid in soup.select("div.jet-listing-grid"):

    print("\nGRID:")
    print("ID:", grid.get("id"))
    print("Classes:", grid.get("class"))

    parent = grid.find_parent(
        attrs={"data-nav": True}
    )

    if parent:
        print("DATA NAV:")
        print(parent.get("data-nav"))


GRID:
ID: None
Classes: ['jet-listing-grid', 'jet-listing']

GRID:
ID: None
Classes: ['jet-listing-grid', 'jet-listing']

GRID:
ID: None
Classes: ['jet-listing-grid', 'jet-listing']


In [20]:
uurl = CATEGORIES["Sante"]

response = session.get(url, timeout=60)
soup = BeautifulSoup(response.text, "lxml")

for i, grid in enumerate(soup.select("div.jet-listing-grid"), 1):

    print(f"\n========== GRID {i} ==========")

    # Inspect the surrounding container
    container = grid.parent

    for _ in range(4):
        if container is None:
            break

        print("\nTAG:", container.name)
        print("ATTRIBUTES:", container.attrs)

        container = container.parent


========== GRID 1 ==========

TAG: div
ATTRIBUTES: {'class': ['elementor-widget-container']}

TAG: div
ATTRIBUTES: {'class': ['elementor-element', 'elementor-element-d6e5248', 'elementor-widget', 'elementor-widget-jet-listing-grid'], 'data-id': 'd6e5248', 'data-element_type': 'widget', 'data-e-type': 'widget', 'data-settings': '{"columns":"5","columns_tablet":"4","columns_mobile":"2"}', 'data-widget_type': 'jet-listing-grid.default'}

TAG: div
ATTRIBUTES: {'class': ['e-con-inner']}

TAG: div
ATTRIBUTES: {'class': ['elementor-element', 'elementor-element-97220d2', 'e-flex', 'e-con-boxed', 'e-con', 'e-parent'], 'data-id': '97220d2', 'data-element_type': 'container', 'data-e-type': 'container', 'data-settings': '{"background_background":"classic"}'}

========== GRID 2 ==========

TAG: div
ATTRIBUTES: {'class': ['elementor-widget-container']}

TAG: div
ATTRIBUTES: {'class': ['elementor-element', 'elementor-element-53faa81', 'elementor-widget', 'elementor-widget-jet-listing-grid'], 'data-i

In [21]:
grid = soup.select("div.jet-listing-grid")[2]

widget = grid.find_parent(
    "div",
    class_="elementor-widget-jet-listing-grid"
)

print(widget.prettify()[:12000])

<div class="elementor-element elementor-element-5a9b631 elementor-widget elementor-widget-jet-listing-grid" data-e-type="widget" data-element_type="widget" data-id="5a9b631" data-settings='{"columns":"5","columns_mobile":"2","columns_tablet":"3"}' data-widget_type="jet-listing-grid.default" id="applyfilter">
 <div class="elementor-widget-container">
  <div class="jet-listing-grid jet-listing">
   <div class="jet-listing-grid__items grid-col-desk-5 grid-col-tablet-3 grid-col-mobile-2 jet-listing-grid--362 jet-equal-columns__wrapper woocommerce" data-listing-id="362" data-listing-source="query" data-nav='{"enabled":true,"type":"scroll","more_el":null,"query":{"query_id":6,"custom_query_id":6,"posts_per_page":10,"allowed_query_ids":[6],"signature":"e0b6aac34f8e2e3a23d77209e4cad0479efef752e797a350fd7f0e3890b62b0e"},"widget_settings":{"lisitng_id":362,"posts_num":10,"columns":5,"columns_tablet":3,"columns_mobile":2,"column_min_width":240,"column_min_width_tablet":240,"column_min_width_mobil

In [25]:
def flatten_payload(data, prefix):
    result = []

    for key, value in data.items():
        new_key = f"{prefix}[{key}]"

        if isinstance(value, dict):
            result.extend(flatten_payload(value, new_key))

        elif isinstance(value, list):
            for item in value:
                result.append((new_key + "[]", str(item)))

        elif isinstance(value, bool):
            result.append((new_key, str(value).lower()))

        else:
            result.append((new_key, str(value)))

    return result


payload = [
    ("action", "jet_engine_ajax"),
    ("handler", "listing_load_more"),
    ("nocache", "1"),
]

payload += flatten_payload(nav["query"], "query")
payload += flatten_payload(nav["widget_settings"], "widget_settings")

payload += [
    ("page_settings[queried_id]", queried_id),
    ("page_settings[page]", "2"),
]

AJAX_URL = f"{BASE_URL}/wp-admin/admin-ajax.php"

ajax_response = session.post(
    AJAX_URL,
    data=payload,
    headers={
        "Referer": CATEGORIES["Sante"],
        "X-Requested-With": "XMLHttpRequest"
    },
    timeout=(10, 60)
)

print("Status:", ajax_response.status_code)
print("Response:", ajax_response.text[:1500])

Status: 200
Response: {"success":true,"data":{"html":"<style>.elementor-362 .elementor-element.elementor-element-94e2b60{--display:flex;--flex-direction:column;--container-widget-width:100%;--container-widget-height:initial;--container-widget-flex-grow:0;--container-widget-align-self:initial;--flex-wrap-mobile:wrap;--justify-content:space-between;--gap:5px 5px;--row-gap:5px;--column-gap:5px;--border-radius:20px 5px 20px 5px;--padding-top:1rem;--padding-bottom:0rem;--padding-left:1rem;--padding-right:1rem;}.elementor-362 .elementor-element.elementor-element-94e2b60:not(.elementor-motion-effects-element-type-background), .elementor-362 .elementor-element.elementor-element-94e2b60 > .elementor-motion-effects-container > .elementor-motion-effects-layer{background-color:#FFFFFF;}.elementor-362 .elementor-element.elementor-element-7901bad .jet-listing-dynamic-image{justify-content:flex-start;}.elementor-362 .elementor-element.elementor-element-7901bad .jet-listing-dynamic-image__figure{align

In [23]:
for i, listing in enumerate(
    soup.select("div.jet-listing-grid__items[data-nav]")
):
    nav = json.loads(listing["data-nav"])

    print(
        "Grid:", i,
        "| Listing:", listing.get("data-listing-id"),
        "| Pages:", listing.get("data-pages"),
        "| Query ID:", listing.get("data-query-id"),
        "| Element ID:", nav.get("widget_settings", {}).get("_element_id")
    )

Grid: 0 | Listing: 764 | Pages: 1 | Query ID: 1 | Element ID: 
Grid: 1 | Listing: 362 | Pages: 1 | Query ID: 10 | Element ID: 
Grid: 2 | Listing: 362 | Pages: 17 | Query ID: 6 | Element ID: applyfilter


In [26]:
listing = soup.select(
    "div.jet-listing-grid__items[data-nav]"
)[2]

nav = json.loads(listing["data-nav"])
queried_id = listing["data-queried-id"]

payload = [
    ("action", "jet_engine_ajax"),
    ("handler", "listing_load_more"),
    ("nocache", "1"),
]

payload += flatten_payload(nav["query"], "query")
payload += flatten_payload(nav["widget_settings"], "widget_settings")

payload += [
    ("page_settings[queried_id]", queried_id),
    ("page_settings[page]", "2"),
]

ajax_response = session.post(
    f"{BASE_URL}/wp-admin/admin-ajax.php",
    data=payload,
    headers={
        "Referer": CATEGORIES["Sante"],
        "X-Requested-With": "XMLHttpRequest"
    },
    timeout=(10, 60)
)

print("Status:", ajax_response.status_code)
print(ajax_response.text[:2000])

Status: 200
{"success":true,"data":{"html":"<style>.elementor-362 .elementor-element.elementor-element-94e2b60{--display:flex;--flex-direction:column;--container-widget-width:100%;--container-widget-height:initial;--container-widget-flex-grow:0;--container-widget-align-self:initial;--flex-wrap-mobile:wrap;--justify-content:space-between;--gap:5px 5px;--row-gap:5px;--column-gap:5px;--border-radius:20px 5px 20px 5px;--padding-top:1rem;--padding-bottom:0rem;--padding-left:1rem;--padding-right:1rem;}.elementor-362 .elementor-element.elementor-element-94e2b60:not(.elementor-motion-effects-element-type-background), .elementor-362 .elementor-element.elementor-element-94e2b60 > .elementor-motion-effects-container > .elementor-motion-effects-layer{background-color:#FFFFFF;}.elementor-362 .elementor-element.elementor-element-7901bad .jet-listing-dynamic-image{justify-content:flex-start;}.elementor-362 .elementor-element.elementor-element-7901bad .jet-listing-dynamic-image__figure{align-items:fle

In [27]:
result = ajax_response.json()

html_page2 = result["data"]["html"]
soup_page2 = BeautifulSoup(html_page2, "lxml")

links_page2 = soup_page2.select(
    "a.jet-listing-dynamic-image__link[href*='/produit/']"
)

urls_page2 = list(dict.fromkeys(
    urljoin(BASE_URL, link["href"])
    for link in links_page2
))

print("Products found on page 2:", len(urls_page2))
print(*urls_page2[:5], sep="\n")

Products found on page 2: 10
https://www.pharmaly-dz.com/produit/cheks-kids-magnesium-calm-60u/
https://www.pharmaly-dz.com/produit/cheks-kids-magnesium-calm-30u/
https://www.pharmaly-dz.com/produit/cheks-glutathione-radiance-ultra-bt-45/
https://www.pharmaly-dz.com/produit/pectino-gummies-pack-hair-boost/
https://www.pharmaly-dz.com/produit/pectino-gummies-junior-sommeil-30u/


In [29]:
import json
import time
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from tqdm import tqdm


def flatten_payload(data, prefix):
    result = []

    for key, value in data.items():
        new_key = f"{prefix}[{key}]"

        if isinstance(value, dict):
            result.extend(flatten_payload(value, new_key))

        elif isinstance(value, list):
            for item in value:
                result.append((new_key + "[]", str(item)))

        elif isinstance(value, bool):
            result.append((new_key, str(value).lower()))

        else:
            result.append((new_key, str(value)))

    return result


def extract_product_urls(html):
    page_soup = BeautifulSoup(html, "lxml")

    urls = [
        urljoin(BASE_URL, link["href"])
        for link in page_soup.select(
            "a.jet-listing-dynamic-image__link[href*='/produit/']"
        )
    ]

    return list(dict.fromkeys(urls))


def add_urls(urls, category, collection):
    for url in urls:
        collection.setdefault(url, [])

        if category not in collection[url]:
            collection[url].append(category)


all_product_urls = {}
failed_pages = []

for category, category_url in tqdm(CATEGORIES.items(), desc="Categories"):

    try:
        response = session.get(category_url, timeout=(10, 60))
        response.raise_for_status()

        category_soup = BeautifulSoup(response.text, "lxml")

        # Find the correct listing using your teammate's configuration
        listing = None
        nav = None

        for candidate in category_soup.select(
            "div.jet-listing-grid__items[data-nav]"
        ):
            candidate_nav = json.loads(candidate["data-nav"])

            if (
                candidate.get("data-listing-id") == "362"
                and candidate_nav.get("widget_settings", {}).get("_element_id")
                == "applyfilter"
                and candidate_nav.get("query", {}).get("query_id") == 6
            ):
                listing = candidate
                nav = candidate_nav
                break

        if listing is None:
            print(f"\nCorrect listing not found: {category}")
            failed_pages.append((category, 1))
            continue

        total_pages = int(listing.get("data-pages", 1))
        queried_id = listing["data-queried-id"]

        # Collect page 1
        first_page_urls = extract_product_urls(str(listing))
        add_urls(first_page_urls, category, all_product_urls)

        print(
            f"\n{category}: {total_pages} pages | "
            f"{len(first_page_urls)} products on page 1"
        )

        # Collect pages 2 to the last page
        for page_number in range(2, total_pages + 1):

            payload = [
                ("action", "jet_engine_ajax"),
                ("handler", "listing_load_more"),
                ("nocache", "1"),
            ]

            payload += flatten_payload(nav["query"], "query")
            payload += flatten_payload(
                nav["widget_settings"], "widget_settings"
            )

            payload += [
                ("page_settings[queried_id]", queried_id),
                ("page_settings[page]", str(page_number)),
            ]

            try:
                ajax_response = session.post(
                    f"{BASE_URL}/wp-admin/admin-ajax.php",
                    data=payload,
                    headers={
                        "Referer": category_url,
                        "X-Requested-With": "XMLHttpRequest"
                    },
                    timeout=(10, 60)
                )

                ajax_response.raise_for_status()
                result = ajax_response.json()

                if not result.get("success"):
                    print(f"  Page {page_number}: AJAX failed")
                    failed_pages.append((category, page_number))
                    continue

                page_urls = extract_product_urls(
                    result["data"]["html"]
                )

                if not page_urls:
                    print(f"  Page {page_number}: no products returned")
                    failed_pages.append((category, page_number))
                    break

                add_urls(page_urls, category, all_product_urls)

                print(
                    f"  Page {page_number}/{total_pages}: "
                    f"{len(page_urls)} products"
                )

                time.sleep(1)

            except (requests.RequestException, ValueError, KeyError) as error:
                print(f"  Page {page_number}: {error}")
                failed_pages.append((category, page_number))

    except requests.RequestException as error:
        print(f"\nCategory failed: {category} | {error}")
        failed_pages.append((category, 1))


print("\n========== COLLECTION SUMMARY ==========")
print("Unique product URLs:", len(all_product_urls))
print("Failed pages:", len(failed_pages))
print("Failed page details:", failed_pages)

Categories:   0%|          | 0/8 [00:00<?, ?it/s]


Sante: 17 pages | 10 products on page 1
  Page 2/17: 10 products
  Page 3/17: 10 products
  Page 4/17: 10 products
  Page 5/17: 10 products
  Page 6/17: 10 products
  Page 7/17: 10 products
  Page 8/17: 10 products
  Page 9/17: 10 products
  Page 10/17: 10 products
  Page 11/17: 10 products
  Page 12/17: 10 products
  Page 13/17: 10 products
  Page 14/17: 10 products
  Page 15/17: 10 products
  Page 16/17: 10 products
  Page 17/17: 10 products


Categories:  12%|█▎        | 1/8 [00:57<06:41, 57.29s/it]


Visage: 261 pages | 10 products on page 1
  Page 2/261: 10 products
  Page 3/261: 10 products
  Page 4/261: 10 products
  Page 5/261: 10 products
  Page 6/261: 10 products
  Page 7/261: 10 products
  Page 8/261: 10 products
  Page 9/261: 10 products
  Page 10/261: 10 products
  Page 11/261: 10 products
  Page 12/261: 10 products
  Page 13/261: 10 products
  Page 14/261: 10 products
  Page 15/261: 10 products
  Page 16/261: 10 products
  Page 17/261: 10 products
  Page 18/261: 10 products
  Page 19/261: 10 products
  Page 20/261: 10 products
  Page 21/261: 10 products
  Page 22/261: 10 products
  Page 23/261: 10 products
  Page 24/261: 10 products
  Page 25/261: 10 products
  Page 26/261: 10 products
  Page 27/261: 10 products
  Page 28/261: 10 products
  Page 29/261: 10 products
  Page 30/261: 10 products
  Page 31/261: 10 products
  Page 32/261: 10 products
  Page 33/261: 10 products
  Page 34/261: 10 products
  Page 35/261: 10 products
  Page 36/261: 10 products
  Page 37/261: 10 pr

Categories:  25%|██▌       | 2/8 [21:15<1:14:01, 740.24s/it]


Corps: 83 pages | 10 products on page 1
  Page 2/83: 10 products
  Page 3/83: 10 products
  Page 4/83: 10 products
  Page 5/83: 10 products
  Page 6/83: 10 products
  Page 7/83: 10 products
  Page 8/83: 10 products
  Page 9/83: 10 products
  Page 10/83: 10 products
  Page 11/83: 10 products
  Page 12/83: 10 products
  Page 13/83: 10 products
  Page 14/83: 10 products
  Page 15/83: 10 products
  Page 16/83: 10 products
  Page 17/83: 10 products
  Page 18/83: 10 products
  Page 19/83: 10 products
  Page 20/83: 10 products
  Page 21/83: 10 products
  Page 22/83: 10 products
  Page 23/83: 10 products
  Page 24/83: 10 products
  Page 25/83: 10 products
  Page 26/83: 10 products
  Page 27/83: 10 products
  Page 28/83: 10 products
  Page 29/83: 10 products
  Page 30/83: 10 products
  Page 31/83: 10 products
  Page 32/83: 10 products
  Page 33/83: 10 products
  Page 34/83: 10 products
  Page 35/83: 10 products
  Page 36/83: 10 products
  Page 37/83: 10 products
  Page 38/83: 10 products
  Pag

... (6 outputs left out)

Categories:  88%|████████▊ | 7/8 [42:08<03:42, 222.82s/it]


Category failed: Hygiene | HTTPSConnectionPool(host='www.pharmaly-dz.com', port=443): Read timed out. (read timeout=60)

Maquillage: 235 pages | 10 products on page 1
  Page 2: Expecting value: line 1 column 1 (char 0)
  Page 3: Expecting value: line 1 column 1 (char 0)
  Page 4: Expecting value: line 1 column 1 (char 0)
  Page 5: Expecting value: line 1 column 1 (char 0)
  Page 6: Expecting value: line 1 column 1 (char 0)
  Page 7: Expecting value: line 1 column 1 (char 0)
  Page 8: Expecting value: line 1 column 1 (char 0)
  Page 9: Expecting value: line 1 column 1 (char 0)
  Page 10: Expecting value: line 1 column 1 (char 0)
  Page 11: Expecting value: line 1 column 1 (char 0)
  Page 12: Expecting value: line 1 column 1 (char 0)
  Page 13: Expecting value: line 1 column 1 (char 0)
  Page 14: Expecting value: line 1 column 1 (char 0)
  Page 15: Expecting value: line 1 column 1 (char 0)
  Page 16: Expecting value: line 1 column 1 (char 0)
  Page 17: Expecting value: line 1 column 1 (

Categories: 100%|██████████| 8/8 [56:50<00:00, 426.34s/it]


========== COLLECTION SUMMARY ==========
Unique product URLs: 6660
Failed pages: 111
Failed page details: [('Solaire', 12), ('Solaire', 13), ('Solaire', 14), ('Solaire', 15), ('Solaire', 16), ('Solaire', 17), ('Solaire', 18), ('Solaire', 19), ('Solaire', 20), ('Solaire', 21), ('Solaire', 22), ('Solaire', 23), ('Solaire', 24), ('Solaire', 25), ('Solaire', 26), ('Solaire', 27), ('Solaire', 28), ('Solaire', 29), ('Solaire', 30), ('Solaire', 31), ('Solaire', 32), ('Solaire', 33), ('Solaire', 34), ('Solaire', 35), ('Solaire', 36), ('Solaire', 37), ('Solaire', 38), ('Solaire', 39), ('Solaire', 40), ('Solaire', 41), ('Solaire', 42), ('Solaire', 43), ('Solaire', 44), ('Solaire', 45), ('Solaire', 46), ('Hygiene', 1), ('Maquillage', 2), ('Maquillage', 3), ('Maquillage', 4), ('Maquillage', 5), ('Maquillage', 6), ('Maquillage', 7), ('Maquillage', 8), ('Maquillage', 9), ('Maquillage', 10), ('Maquillage', 11), ('Maquillage', 12), ('Maquillage', 13), ('Maquillage', 14), ('Maquillage', 15), ('Maquill

In [30]:
urls_df = pd.DataFrame([
    {
        "product_url": url,
        "categories": ", ".join(categories)
    }
    for url, categories in all_product_urls.items()
])

urls_df.to_csv(
    "../data/product_urls.csv",
    index=False,
    encoding="utf-8-sig"
)

display(urls_df.head())
print("Saved:", len(urls_df), "unique product URLs")

,product_url,categories
0,https://www.pharmaly-dz.com/produit/biogaia-go...,Sante
1,https://www.pharmaly-dz.com/produit/fortimel-s...,Sante
2,https://www.pharmaly-dz.com/produit/acm-novoph...,Sante
3,https://www.pharmaly-dz.com/produit/cheks-post...,Sante
4,https://www.pharmaly-dz.com/produit/granions-b...,Sante


Saved: 6660 unique product URLs
